In [ ]:
modello_per_analisi = "Opel_Corsa"
comune_per_analisi = "Sant'Agata Bolognese"
prezzo_minimo = 3000
prezzo_massimo = 30000
km_minimi = 0
km_massimi = 0
immatricolazione_minima = 2020
immatricolazione_massima = 2025

## 0. Import funzioni e librerie

In [ ]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import requests
import os
import re
import importlib
from pathlib import Path
import sys
import time

In [ ]:
from vehicle_price_monitor.paths import ROOT, RAW, PROCESSED, MODELS, GEO, load_config
project_root = ROOT

In [ ]:
#Scraping
from vehicle_price_monitor.scraping.url_builders import build_automobile_url, build_subito_url, build_autosupermarket_url, build_autotorino_url, bild_autoscout_urls
from vehicle_price_monitor.scraping.scraping_functions import autoscout_scraper, autosupermarket_scraper, automobile_it_scraper, subito_scraper
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import random

In [ ]:
config = load_config(modello_per_analisi)
# --- Accesso ai parametri di configurazione ---
#allestimenti = config.allestimenti
allestimento_performance = config.allestimento_performance
allestimento_sport = config.allestimento_sport
allestimento_middle = config.allestimento_middle
allestimento_base = config.allestimento_base
motorizzazioni_norm = config.motorizzazioni_norm
modelli_ord = config.modelli_ord
mappa_cv = config.mappa_cv
mappa_allestimenti = config.mappa_allestimenti

## 1. Creazione e importazione URLs

In [ ]:
marca, modello = [p.lower() for p in modello_per_analisi.split("_")]

In [ ]:
# ------------------- USO DELLE FUNZIONI: Urls_building -------------------
url_asm = build_autosupermarket_url(marca, modello, prezzo_minimo, prezzo_massimo,
                                    km_minimi, km_massimi, immatricolazione_minima, immatricolazione_massima)

url_at = build_automobile_url(marca, modello, prezzo_minimo, prezzo_massimo,
                              km_minimi, km_massimi, immatricolazione_minima, immatricolazione_massima)

url_su = build_subito_url(marca, modello, prezzo_minimo, prezzo_massimo,
                              km_minimi, km_massimi, immatricolazione_minima, immatricolazione_massima)

url_ar = build_autotorino_url(marca_sito = 818, #codice di autotorino per Opel
                              modello_sito = 1028, #codice di autotorino per Corsa
                              prezzo_minimo=prezzo_minimo,prezzo_massimo=prezzo_massimo,km_minimi=km_minimi,km_massimi=km_massimi,
                              anno_min=immatricolazione_minima,anno_max=immatricolazione_massima)

urls_as = bild_autoscout_urls (marca, modello, prezzo_minimo, prezzo_massimo,
                              km_minimi, km_massimi, immatricolazione_minima, immatricolazione_massima)

# ------------------- STAMPA DEGLI URL -------------------
print("Automobile.it URL:", url_at)
print("Autosupermarket URL:", url_asm)
print("Subito.it URL:", url_su)
print("Autotorino URL:", url_ar)
print("Autoscout URLs:", urls_as)

## 2. Scraping dei dati

### 2.1. Scraping - Autoscout

In [ ]:
# Lista per raccogliere tutti i DataFrame
all_dataframes = []

# Ciclo su tutti gli URL
for i, url in enumerate(urls_as, start=1):
    print(f"\n🔍 Scraping URL {i}/{len(urls_as)}: {url}")
    try:
        df_temp = autoscout_scraper(url_template=url + "&page={}", max_pages=20)
        if df_temp is not None and not df_temp.empty:
            df_temp['Categoria'] = f'URL_{i}'  # colonna per sapere da quale URL proviene
            all_dataframes.append(df_temp)
        else:
            print(f"⚠ Nessun dato trovato per URL {i}")
    except Exception as e:
        print(f"❌ Errore durante lo scraping dell'URL {i}: {e}")
        continue  # passa al prossimo URL anche se c'è un errore

# Concatenazione dei tutti i DataFrame in uno solo
if all_dataframes:
    df_as = pd.concat(all_dataframes, ignore_index=True)
    print("\n✅ Raccolti tutti i dati!")
    print(df_as.head())
else:
    print("❌ Nessun dato trovato da nessun URL.")

In [ ]:
"""
# Lista per raccogliere tutti i DataFrame
all_dataframes = []

# Ciclo su tutti gli URL
for i, url in enumerate(urls_as, start=1):
    print(f"\n🔍 Scraping URL {i}/{len(urls_as)}...")
    df_temp = autoscout_scraper(url_template=url + "&page={}", max_pages=20)
    if df_temp is not None and not df_temp.empty:
        df_temp['Categoria'] = f'URL_{i}'  #colonna per sapere da quale URL proviene
        all_dataframes.append(df_temp)
    else:
        print(f"⚠ Nessun dato trovato per URL {i}")

# Concatenazione dei tutti i DataFrame in uno solo
if all_dataframes:
    df_as = pd.concat(all_dataframes, ignore_index=True)
    print("\n✅ Raccolti tutti i dati!")
    print(df_as.head())
else:
    print("❌ Nessun dato trovato da nessun URL.")
    
"""

In [ ]:
print(df_as.info())
df_as.head()

### 2.2. Scraping - Automobili.it

In [ ]:
df_at = automobile_it_scraper(url_at)

In [ ]:
df_at

In [ ]:
df_at.info()

### 2.3. Scraping - Autosupermarket

In [ ]:
df_asm = autosupermarket_scraper(url_asm)

In [ ]:
df_asm['Prezzo'] = df_asm['Prezzo'].apply(lambda x: x.split('\n')[-1] if '%' in x else x)

In [ ]:
df_asm

In [ ]:
for col in df_asm.columns:
    #print(f"{col}: {df_asm[col].isnull().sum()} valori mancanti")
    #print(f"{col}: {df_asm[col].unique()} valori unici")
    print(f"- {col}: {(df_asm[col] == '').sum()} somma dei valori vuoti")

In [ ]:
df_asm.replace('', np.nan, inplace=True)
df_asm.replace('Prezzo riservato', np.nan, inplace=True)

In [ ]:
df_asm.info()

In [ ]:
df_asm.dropna(inplace=True)    

### 2.4. Scraping - Subito.it

In [ ]:
df_su = subito_scraper(url_su)

In [ ]:
for col in df_su.columns:
    print(f"- {col}: {(df_su[col] == '').sum()} somma dei valori vuoti")

In [ ]:
df_su.info()

In [ ]:
df_su

### 2.5. Scraping Autotorino (da realizzare)

### 2.6. Export dei dati

In [ ]:
# Percorsi delle cartelle contenenti i moduli
output_folder_df_grezzi = RAW / modello_per_analisi


#Verifica se la cartella esiste, se no la crea
output_folder_df_grezzi.mkdir(parents=True, exist_ok=True)

In [ ]:
def salva_df(df, output_path, modello):
    if df is not None and not df.empty:
        df.to_csv(output_path, index=False)
        print(f"""💾 File salvato, del modello {modello},
    nel percorso: {output_path}""")
    else:
        print(f"⚠️ Nessun dato da salvare per il modello {modello} in {output_path}.")

In [ ]:
try:
    salva_df(df_as.drop(columns='Categoria', errors='ignore'), output_folder_df_grezzi / "data_autoscout.csv", modello_per_analisi)
except Exception as e:
    print(f"⚠️ Errore con df_as: {e}")

try:
    salva_df(df_at, output_folder_df_grezzi / "data_automobile_it.csv", modello_per_analisi)
except Exception as e:
    print(f"⚠️ Errore con df_at: {e}")

try:
    salva_df(df_asm, output_folder_df_grezzi / "data_autosupermarket.csv", modello_per_analisi)
except Exception as e:
    print(f"⚠️ Errore con df_asm: {e}")

try:
    salva_df(df_su, output_folder_df_grezzi / "data_subito_it.csv", modello_per_analisi)
except Exception as e:
    print(f"⚠️ Errore con df_su: {e}")

'''
try:
    salva_df(df_ar, output_folder_df_grezzi / "data_autotorino.csv", modello_per_analisi)
except Exception as e:
    print(f"⚠️ Errore con df_ar: {e}")
    
'''


## 3. Data Preparation

### 3.1. Import dei dataframe

In [ ]:
#Import dati relativi ai comuni e alla distanza
distanza = pd.read_csv(f'{GEO}/distanza.csv')
only_cap_per_comune = pd.read_csv(f'{GEO}/only_cap_per_comune.csv')
only_comune_per_cap = pd.read_csv(f'{GEO}/only_comune_per_cap.csv')

In [ ]:
# Import dati grezzi
data_at = pd.read_csv(f'{RAW}/{modello_per_analisi}/data_automobile_it.csv')
data_as = pd.read_csv(f'{RAW}/{modello_per_analisi}/data_autoscout.csv')
data_asm = pd.read_csv(f'{RAW}/{modello_per_analisi}/data_autosupermarket.csv')
data_su = pd.read_csv(f'{RAW}/{modello_per_analisi}/data_subito_it.csv')

### 3.2. Pulizia singoli DF

#### 3.2.1. Autoscout

In [ ]:
#Pre pulizia
print(data_as.info())
data_as.head(4)

In [ ]:
data_AS = clean_data_AS(data_as, only_comune_per_cap, motorizzazioni_norm, modelli_ord, mappa_cv)
print(data_AS.info())
data_AS.head(15)

#### 3.2.2. Automobile.it

In [ ]:
print(data_at.info())
data_at.head(4)

In [ ]:
data_AT = clean_data_AT(data_at, only_cap_per_comune, motorizzazioni_norm, modelli_ord, mappa_cv)
print(data_AT.info())
data_AT.head(4)

In [ ]:
print(data_asm.info())
data_asm.head(4)

#### 3.2.3. Autosupermarket

In [ ]:
data_ASM = clean_data_ASM(data_asm, motorizzazioni_norm, modelli_ord)
print(data_ASM.info())
data_ASM.head(4)

In [ ]:
print(data_su.info())
data_su.head(4)

#### 3.2.4. Subito.it

In [ ]:
data_SU = clean_data_SU(data_su, only_cap_per_comune, motorizzazioni_norm, modelli_ord, mappa_cv)
print(data_SU.info())
data_SU.head(4)

### 3.3. Formattazione Dataset finale

In [ ]:
data = pd.concat([data_AS, data_AT, data_ASM, data_SU
                  #data_AR
                  ], ignore_index=True)
# Output diagnostico per verificare l'unione
print("\n✅ Dataset unificato caricato correttamente.")
print(f"Numero di righe: {data.shape[0]}")
print(f"Numero di colonne: {data.shape[1]}")
print("\n📋 Informazioni dettagliate per il dataframe NON FORMATTATO:")
data.info()
print()

In [ ]:
data['Venditore']=data['Venditore'].fillna("Rivenditore")

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data_formattato = data_formatting(data, only_cap_per_comune, distanza, comune_per_analisi, mappa_allestimenti, allestimento_performance, allestimento_sport, allestimento_middle, allestimento_base)
print("\n📋 Informazioni dettagliate per il dataframe FORMATTATO:")
data_formattato.info()

In [ ]:
data_formattato[data_formattato['Chilometraggio']>200000]

In [ ]:
# Visualizzazione dei valori unici delle colonne categoriali (escluse alcune colonne informative)
exclude_columns = ["località", "CAP", "Comune", "Annuncio", "Link", "Regione"]
object_columns = [
    col for col in data_formattato.select_dtypes(include='object').columns 
    if col not in exclude_columns]
for column in object_columns:
    print("\nValori unici delle variabili Categoriali, riferite ai veicoli:")
    print(f"Colonna: {column}: {data_formattato[column].unique()}")
    print("-" * 40)

### 3.4. Dataset dummy

In [ ]:
# --- 6. CREAZIONE DF DUMMY X ANALISI ---
data_dummy = get_data_dummy(data_formattato, allestimento_performance, allestimento_sport, allestimento_middle)
print("\n📋 Informazioni per il dataframe DUMMY:")
data_dummy.info()

### 3.5. Salvataggio dei dati

In [ ]:
output_folder_df = PROCESSED / modello_per_analisi
output_folder_df.mkdir(parents=True, exist_ok=True)

print("Cartella di salvataggio:", output_folder_df)
data.to_csv(os.path.join(output_folder_df, f"data_NF_{modello_per_analisi}NF.csv"), index=False)
data_formattato.to_csv(os.path.join(output_folder_df, f"data_{modello_per_analisi}.csv"), index=False)
data_dummy.to_csv(os.path.join(output_folder_df, f"data_dummy_{modello_per_analisi}.csv"), index=False)
print("\n✅ Programma completato con successo! I dataset sono stati salvati nella cartella:\n", output_folder_df)